# Argon A-to-Z

This tutorial demonstrates a-to-z how to optimise Lennard Jones parameters for liquid argon, and without going into details. For details see other tutorials and wider MDMC documentation.

In [1]:
# Imports used for this tutorial
import numpy as np
import os
from MDMC.control import Control
from MDMC.MD import Atom, Dispersion, LennardJones, Simulation, Universe

Supported DL_POLY version 5.0


In [2]:
# Change the number of threads depending on the number of physical cores on your computer
# as it was tested for LAMMPS
os.environ["OMP_NUM_THREADS"] = "8"

In [3]:
# Build universe with density 0.0176 atoms per AA^-3
density = 0.0176
# This means cubic universe of side:
# 23.0668 A will contain 216 Ar atoms
# 26.911 A will contain 343 Ar atoms
# 30.7553 A will contain 512 Ar atoms
# 38.4441 A will contain 1000 Ar atoms
universe = Universe(dimensions=30.7553)
Ar = Atom('Ar', charge=0., mass=36.0)
# Calculating number of Ar atoms needed to obtain density
n_ar_atoms = int(density * np.product(universe.dimensions))
print(f'Number of argon atoms = {n_ar_atoms}')
universe.fill(Ar, num_struc_units=(n_ar_atoms))

Universe created with:
  Dimensions       [30.76, 30.76, 30.76]
  Force field                       None
  Number of atoms                      0

Number of argon atoms = 512


In the Jupyter cell above, a box of Argon atoms is set up. However, at this point there is no interaction forces between the argon atoms! In the cell below an appropriate (for argon) force-field interaction potential is defined.

In [4]:
Ar_dispersion = Dispersion(universe,
                           (Ar.atom_type, Ar.atom_type),
                           cutoff=8.,
                           function=LennardJones(epsilon=1.0, sigma=3.36))

In this case the interaction potential chosen is the humble Lennard Jones (to get info see doc or type `help(LennardJones)`).

Also, a `cutoff` value is chosen (see `help(Dispersion)` for more info). A [rule of thumb for Lennard-Jones](https://en.wikipedia.org/wiki/Lennard-Jones_potential) is to pick `cutoff=2.5*sigma`. The value for argon is recommended to be between 8 and 12 ang. `cutoff` is not a force-field parameter and therefore will not be refined. Ideally, and for any system you want to pick at value of the `cutoff` which is small while not compromising accuracy. For this system picking a value between 8 and 12 ang is found to give near identifical results.

Next (and before starting the refinement), we set up the MD engine and equilibrate the system. Note with MDMC the equilibration only needs to be done once. 

In [9]:
# MD Engine setup
simulation = Simulation(universe,
                        engine="lammps",
                        time_step=6.582160439079001,
                        temperature=120.,
                        traj_step=10)

LAMMPS (29 Sep 2021 - Update 3)
  using 8 OpenMP thread(s) per MPI task
LAMMPS output is captured by PyLammps wrapper
LAMMPS (29 Sep 2021 - Update 3)
  using 8 OpenMP thread(s) per MPI task
LAMMPS output is captured by PyLammps wrapper
Total wall time: 0:00:00
using multi-threaded neighbor list subroutines
Simulation created with lammps engine and settings:
  temperature  120.0



In [10]:
# Energy Minimization and equilibration
simulation.minimize(n_steps=5000)
simulation.run(n_steps=10000, equilibration=True)

OK; time to set up the actual refinement of the force-field parameters. 

First we need some data to refine against:

In [11]:
# exp_datasets is a list of dictionaries with one dictionary per experimental
# dataset
# Dataset from: van Well et al. (1985). Physical Review A, 31(5), 3391-3414
# resolution is None as the original author already accounted for instrument resolution
exp_datasets = [{'file_name':'data/exported_sqw.csv',
                 'type':'SQw',
                 'reader':'MantidSQw',
                 'weight':1.,
                 'auto_scale':True,
                 'resolution':800.}]

The number of `MD_steps` specified must be large enough to allow for successful calculation of all observables. This depends the `type` of the dataset provided and the value of the `traj_step` (specified when creating the `Simulation`). If a value for `MD_steps` is not provided, then the minimum number needed will be used automatically.

Additionally, some observables will have an upper limit on the number of MD_steps that can be used in calculating their dependent variable(s). In these cases, the number of `MD_steps` is rounded down to a multiple of this upper limit so that we only run steps that will be useful. For example, if we use 1000 `MD_steps` in calculation, but a value of 2500 is provided, then we will run 2000 steps and use this to calculate the variable twice, without wasting time performing an additional 500 steps.

In [13]:
fit_parameters = universe.parameters
fit_parameters['sigma'].constraints = [2.8,3.9]
fit_parameters['epsilon'].constraints = [0.5, 1.4]


control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  minimizer_type="GPO",
                  reset_config=True,
                  MD_steps=19980,
                  equilibration_steps=4000,
                  n_initial = 21)

Control created with:
- Attributes                              -
  Minimizer                             GPO
  FoM type               ChiSquaredExpError
  Number of observables                   1
  Number of parameters                    2



And finally start the refinement! Bump up `n_steps` from 3 when you are ready.

In [16]:
control.FoM_calculator.value

nan

In [14]:
# Run the refinement, i.e. refine the FF parameters against the data
control.refine(n_steps=25)
#control.plot_results();

Step         FoM Change state  Pred coords     Pred FoM epsilon (#2)   sigma (#3)
Total wall time: 0:02:29


AssertionError: 

In [ ]:
control.plot_results();

In [ ]:
from skopt.plots import plot_evaluations, plot_objective

In [ ]:
_ = plot_evaluations(control.minimizer.optimizer.get_result(), bins=10)
_ = plot_objective(control.minimizer.optimizer.get_result(), n_samples=30, dimensions=['epsilon', 'sigma'])

In [ ]:
obs_pair_lmp = control.observable_pairs[0]

print(obs_pair_lmp.exp_obs.data)
print(obs_pair_lmp.MD_obs.data)

help(obs_pair_lmp)

In [ ]:
obs_pair_lmp.rescale_factor


In [ ]:
result_lmp = obs_pair_lmp.MD_obs.SQw*obs_pair_lmp.rescale_factor

In [ ]:
obs_pair_lmp = control.observable_pairs[0]
result_lmp = obs_pair_lmp.MD_obs.SQw/obs_pair_lmp.rescale_factor

%matplotlib widget
from MDMC.trajectory_analysis.observables.obs_factory import ObservableFactory
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
obs=ObservableFactory.create_observable('SQw')
obs.read_from_file(reader='xml_SQw', file_name='data/Well_s_q_omega_Ar_data.xml')
SQw=obs.SQw[0]
SQw_err=obs.SQw_err[0]
Q=obs.Q
E=obs.E
fig, ax = plt.subplots()
line, = ax.plot(E, SQw[1], linewidth=1, color='black')
SQw_lmp = result_lmp
line_lmp, = ax.plot(E, SQw_lmp[0,1,:], linewidth=1, color='blue')
# ax.set_xlim([-1, 1])
ax.set_xlabel('E (meV)')
ax.set_ylabel('S(Q,E) (arb)')
ax.set_title('Argon data')
fig.subplots_adjust(left=0.25, bottom=0.3)
Q_slider_ax  = fig.add_axes([0.25, 0.15, 0.65, 0.03], facecolor='lightgoldenrodyellow')
Q_slider = Slider(Q_slider_ax, 'Q index', 0, len(Q)-1, valinit=1, valstep=1)
Q_label=plt.text(1,1.7,f'Q={Q[1]} $\AA^{-1}$')
def Q_on_changed(val):
    line.set_ydata(SQw[val])
    line_lmp.set_ydata(SQw_lmp[0,val])
    Q_label.set_text(f'Q={Q[val]} $\AA^{-1}$')
    fig.canvas.draw_idle()
    ax.set_ylim(0,max(np.max(SQw[:,val]),1e-5))
    #ax.set_yscale('log')
Q_slider.on_changed(Q_on_changed)
plt.show()